# Driven, damped oscillator: resonance

Here, we re-do some of Taylor's figures via interactive plots.

**Same system, two knobs.** Both resonance plots are the same formula,
$A^2 = \dfrac{f_0^2}{(\omega_0^2 - \omega^2)^2 + 4\beta^2\omega^2}$, sliced two ways.
Figure 5.16 turns $\omega_0$ with the drive $\omega$ fixed (tuning a radio: the station is fixed, you retune the circuit).
Figure 5.17 turns $\omega$ with $\omega_0$ fixed (shaking one oscillator at different rates).
So the peaks come from two different derivatives, $d/d\omega_0$ and $d/d\omega$, and they are not the same.
The damping term $4\beta^2\omega^2$ does not care about $\omega_0$, so for $d/d\omega_0$ it is a constant and Figure 5.16 peaks exactly at $\omega_0 = \omega$.
For $d/d\omega$ it grows with $\omega$ (faster shaking, faster motion, more friction), so it pays to drive a little slow: Figure 5.17 peaks at $\omega_2 = \sqrt{\omega_0^2 - 2\beta^2}$, just below $\omega_0$.
The inset on each plot zooms in on its top so you can see the difference; for small $\beta$ it is tiny ($\omega_2 = 0.990$ at $\omega_0 = 1$, $\beta = 0.1$).

Top left: Taylor's Figure 5.16, $A^2$ against the natural frequency $\omega_0$, drive $\omega$ fixed.
Top right: Taylor's Figure 5.17, $A^2$ against the drive frequency $\omega$, $\omega_0$ fixed at the slider's value.
Bottom: Taylor's Figure 5.15(b), the motion $x(t)$ that the drive $f(t) = f_0\cos\omega t$ produces, starting from $x_0$ with velocity $v_0$.
The clock beside it is the phase lag $\delta$: picture both $f(t)$ and the steady motion as hands turning counterclockwise at $\omega$; the motion's hand (purple) trails the drive's (black) by $\delta$: about 0 when $\omega_0 \gg \omega$, a quarter turn at $\omega_0 = \omega$, nearly half a turn when $\omega_0 \ll \omega$.
The purple dot is the system you have set up, on both resonance plots: same $A^2$, two different slices through it.

For convenience, we fix the drive at $\omega = 1$ rad/s (period $2\pi$ s) and $f_0 = 1$. The sliders $\omega_0$ and $\beta$ are in rad/s and 1/s; $x_0$ and $v_0$ set the initial conditions.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

omega = 1.0                # drive frequency (rad/s)
f0 = 1.0                   # drive strength
t = np.linspace(0, 50, 2000)        # seconds: about 8 drive periods
w0s = np.linspace(0.01, 3, 2000)    # natural frequencies for Figure 5.16
ws = np.linspace(0.01, 3, 2000)     # drive frequencies for Figure 5.17

## The two figures

The steady state is $x_p = A\cos(\omega t - \delta)$ with
$A^2 = \dfrac{f_0^2}{(\omega_0^2 - \omega^2)^2 + (2\beta\omega)^2}$ and $\delta = \tan^{-1}\dfrac{2\beta\omega}{\omega_0^2 - \omega^2}$ (Taylor 5.64-5.65).
The transient $x_h$ is last class's damped solution, with constants chosen so that $x_h + x_p$ starts at $x_0$, $v_0$.

In Figure 5.16 the peak sits at $\omega_0 = \omega$ with height $A^2_{\max} = (f_0/2\beta\omega)^2$. Half of that is reached where $(\omega_0^2 - \omega^2)^2 = (2\beta\omega)^2$, i.e. $\omega_0^2 = \omega^2 \pm 2\beta\omega$, so the full width at half maximum is FWHM $\approx 2\beta$ when $\beta \ll \omega$ (the same as when $\omega$ is the knob; Taylor p. 189). The red arrow on each resonance plot spans the exact half-maximum points; its label gives $2\beta$.

Two things on Figure 5.16 that look like typos but are not.
$A^2$ does not go to zero at $\omega_0 = 0$: no spring ($k = 0$) leaves a free, damped mass being shaken back and forth, and it still moves, with $A^2 = f_0^2/(\omega^4 + 4\beta^2\omega^2)$ ($1/(1 + 4\beta^2)$ here, close to 1).
And the peak is lopsided: $A^2$ depends on $\omega_0$ only through $\omega_0^2$, and as a function of $\omega_0^2$ it is symmetric about $\omega^2$ (the half-maximum points $\omega^2 \pm 2\beta\omega$ are equally spaced). Taking the square root to plot against $\omega_0$ pulls the lower half-maximum point farther from $\omega$ than the upper one: at $\beta = 0.5$ they are at 0 and 1.41.
The top itself is exactly at $\omega_0 = \omega$: $d/d\omega_0$ of the denominator is $4\omega_0(\omega_0^2 - \omega^2)$.

In [ ]:
def A_and_delta(w0, beta, w=omega):
    A = f0 / np.sqrt((w0**2 - w**2)**2 + (2 * beta * w)**2)
    delta = np.arctan2(2 * beta * w, w0**2 - w**2)    # arctan2 keeps delta between 0 and pi
    return A, delta

def x_h(xh0, vh0, beta, w0, t):
    # last class's damped solution starting at xh0, vh0 (any damping)
    if np.isclose(beta, w0):                                  # critically damped
        return (xh0 + (vh0 + beta * xh0) * t) * np.exp(-beta * t)
    q = np.sqrt(complex(beta**2 - w0**2))
    C1 = (vh0 + (beta + q) * xh0) / (2 * q)
    C2 = -(vh0 + (beta - q) * xh0) / (2 * q)
    return (np.exp(-beta * t) * (C1 * np.exp(q * t) + C2 * np.exp(-q * t))).real

def mark_fwhm(ax, lo, hi, half, beta):
    ax.annotate('', xy=(lo, half), xytext=(hi, half),
                arrowprops=dict(arrowstyle='<->', color='red', shrinkA=0, shrinkB=0), annotation_clip=True)
    label = rf'FWHM $\approx 2\beta = {2 * beta:.2f}$'
    if hi < 2:
        ax.text(hi + 0.05, half, label, color='red', va='top', clip_on=True)
    else:    # no room on the right: put it on the left
        ax.text(lo - 0.05, half, label, color='red', va='top', ha='right', clip_on=True)

def zoom_top(ax, xs, ys, ref, ref_label, top, beta, left_side):
    """Inset on the top of the curve: gray line at ref, dashed purple line at the actual peak."""
    lo, hi = max(min(ref, top) - 0.4 * beta, 0), max(ref, top) + 0.4 * beta
    keep = (xs >= lo) & (xs <= hi)
    box = ax.inset_axes([0.12, 0.58, 0.36, 0.38] if left_side else [0.61, 0.58, 0.36, 0.38])
    box.plot(xs[keep], ys[keep], 'k')
    box.axvline(ref, color='gray', lw=1, label=ref_label)
    box.axvline(top, color='purple', ls='--', lw=1, label=f'peak, {top:.3f}')
    box.set_xlim(lo, hi)
    box.set_ylim(ys[keep].min(), ys[keep].max() * 1.02)
    box.tick_params(labelsize=7)
    box.legend(fontsize=7, loc='lower center', frameon=False)

def phase_clock(clock, delta):
    """Clock: the drive's hand and the steady motion's hand, which is delta behind it.
    Both hands turn counterclockwise at omega; behind means clockwise from the drive."""
    clock.set_aspect('equal'); clock.axis('off')
    a = np.linspace(0, 2 * np.pi, 200)
    clock.plot(np.cos(a), np.sin(a), color='lightgray', lw=1)
    arc = np.linspace(0, -delta, 50)
    clock.plot(0.35 * np.cos(arc), 0.35 * np.sin(arc), color='red', lw=1)
    clock.annotate('', xy=(1, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    clock.annotate('', xy=(np.cos(delta), -np.sin(delta)), xytext=(0, 0),
                   arrowprops=dict(arrowstyle='->', color='purple', lw=2))
    clock.text(1.08, 0.08, 'drive', fontsize=10, va='bottom')
    clock.text(1.15 * np.cos(delta), -1.15 * np.sin(delta), 'motion', color='purple', fontsize=10,
               ha='left' if np.cos(delta) > 0 else 'right', va='top')
    clock.set_xlim(-1.6, 1.9); clock.set_ylim(-1.4, 1.4)
    clock.set_title(rf'$\delta = {delta:.2f}$ rad, {delta / (2 * np.pi):.2f} cycle' + '\n' + 'motion behind drive', fontsize=10, color='red')

def plot_resonance(w0=1.5, beta=0.1, x0=0.0, v0=0.0, fixed_axes=True):
    A, delta = A_and_delta(w0, beta)
    xp = A * np.cos(omega * t - delta)
    # the transient makes up the difference between where x_p starts and where the cart starts
    x = xp + x_h(x0 - A * np.cos(delta), v0 - A * omega * np.sin(delta), beta, w0, t)

    fig = plt.figure(figsize=(13, 9))
    grid = fig.add_gridspec(2, 2, height_ratios=[4, 3], hspace=0.35)
    fig16, fig17 = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])
    bottom = grid[1, :].subgridspec(1, 2, width_ratios=[4, 1], wspace=0.05)
    motion, clock = fig.add_subplot(bottom[0]), fig.add_subplot(bottom[1])

    # Figure 5.16: A^2 against omega_0 at fixed omega; peak exactly at omega_0 = omega
    A2_16 = A_and_delta(w0s, beta)[0]**2
    peak16 = (f0 / (2 * beta * omega))**2
    fig16.plot(w0s, A2_16, 'k')
    fig16.plot([w0], [A**2], 'o', color='purple', markersize=10)
    fig16.axvline(omega, color='gray', lw=0.5)
    fig16.set_xlabel(r'$\omega_0$ (rad/s)'); fig16.set_ylabel(r'$A^2$')
    fig16.set_title(rf'Fig. 5.16: turn $\omega_0$, $\omega = {omega:g}$ fixed' + '\n'
                    + rf'peak $= {peak16:.1f}$ at $\omega_0 = \omega$ exactly')
    # half the peak where omega_0^2 = omega^2 -/+ 2 beta omega
    mark_fwhm(fig16, np.sqrt(omega**2 - 2 * beta * omega), np.sqrt(omega**2 + 2 * beta * omega), peak16 / 2, beta)
    zoom_top(fig16, w0s, A2_16, omega, r'$\omega$', omega, beta, left_side=False)

    # Figure 5.17: A^2 against omega at fixed omega_0; peak at omega_2 = sqrt(omega_0^2 - 2 beta^2)
    A2_17 = A_and_delta(w0, beta, ws)[0]**2
    fig17.plot(ws, A2_17, 'k')
    fig17.plot([omega], [A**2], 'o', color='purple', markersize=10)
    fig17.axvline(w0, color='gray', lw=0.5)
    fig17.set_xlabel(r'$\omega$ (rad/s)'); fig17.set_ylabel(r'$A^2$')
    if w0**2 > 2 * beta**2:
        w2 = np.sqrt(w0**2 - 2 * beta**2)
        peak17 = f0**2 / (4 * beta**2 * (w0**2 - beta**2))
        # half the peak where omega^2 = omega_2^2 -/+ 2 beta sqrt(omega_0^2 - beta^2)
        u_lo, u_hi = w2**2 - 2 * beta * np.sqrt(w0**2 - beta**2), w2**2 + 2 * beta * np.sqrt(w0**2 - beta**2)
        if u_lo > 0:
            mark_fwhm(fig17, np.sqrt(u_lo), np.sqrt(u_hi), peak17 / 2, beta)
        where = rf'$\omega_2 = {w2:.3f}$, just below $\omega_0 = {w0:.2f}$'
    else:   # 2 beta^2 >= omega_0^2: no peak away from omega = 0
        w2, peak17 = 0.0, f0**2 / w0**4
        where = r'$\omega = 0$ ($\beta \geq \omega_0/\sqrt{2}$)'
    fig17.set_title(rf'Fig. 5.17: turn $\omega$, $\omega_0 = {w0:.2f}$ fixed' + '\n'
                    + rf'peak $= {peak17:.1f}$ at ' + where)
    zoom_top(fig17, ws, A2_17, w0, r'$\omega_0$', w2, beta, left_side=w0 > 1.6)

    # Figure 5.15(b): the motion
    motion.plot(t, x, 'purple', label='x(t)')
    motion.plot(t, xp, color='gray', ls='--', lw=1, label='steady state')
    motion.axhline(0, color='gray', lw=0.5)
    motion.set_xlabel('t (s)'); motion.set_ylabel('x')
    motion.set_title(rf'Fig. 5.15(b): the motion, $\omega_0 = {w0:.2f}$, $\beta = {beta:.2f}$, drive $\omega = {omega:g}$')
    motion.legend(loc='upper right', frameon=False)
    phase_clock(clock, delta)

    if fixed_axes:   # fixed scales, so growing and shrinking show up as growing and shrinking
        fig16.set_ylim(0, 40); fig17.set_ylim(0, 40)
        motion.set_ylim(-6.5, 6.5)
        if peak17 > 40:
            fig17.text(0.65, 0.3, 'peak is off this scale:\nuntick fixed_axes', transform=fig17.transAxes,
                       ha='center', color='gray')
    plt.show()

interact(plot_resonance,
         w0=FloatSlider(value=1.5, min=0.2, max=3, step=0.05, description='ω0'),
         beta=FloatSlider(value=0.1, min=0.08, max=0.5, step=0.01, description='β'),
         x0=FloatSlider(value=0.0, min=-2, max=2, step=0.1, description='x0'),
         v0=FloatSlider(value=0.0, min=-2, max=2, step=0.1, description='v0'),
         fixed_axes=True);

## Things to try

- Start at $\omega_0 = 1.5$ and slide toward 1. Then past it.
- At $\omega_0 = 1$, change $\beta$ from 0.1 to 0.2. The peak height should drop by a factor of 4 (it goes as $1/\beta^2$), and the red FWHM arrow doubles in length.
- Far from resonance ($\omega_0 = 3$) the steady motion is tiny on the fixed scale: untick `fixed_axes` to zoom in.
- Watch the messy start: while $\beta < \omega_0$, how long it lasts depends on $\beta$ only, not on $\omega_0$ (it dies as $e^{-\beta t}$).
- The messiest start, most like Taylor's Figure 5.15: $\omega_0 = 3$ with `fixed_axes` unticked.
- Push $\beta$ up to 0.5: FWHM $\approx 2\beta$ needs $\beta \ll \omega$, and the arrow (the exact width, 1.41 at $\beta = 0.5$) pulls away from $2\beta$.
- Compare the insets at $\omega_0 = 1$: at $\beta = 0.1$ Figure 5.16's peak sits on its gray line and Figure 5.17's is 1% below it. At $\beta = 0.4$ Figure 5.16's still does; Figure 5.17's has moved to $\omega_2 = 0.825$.

*(your notes here)*